# Class 05 — Documents, Metadata, and Vector Search with LangChain

## Exercise 1 — Building Documents by Hand

Before automating the migration of the Class 04 chunks, this exercise builds
some LangChain `Document` objects by hand to understand the structure:
`page_content` (the text) and `metadata` (a free-form dictionary) — with no
embedding field, since that's the vector store's responsibility, not the
`Document`'s.

In [2]:
!pip install -q langchain-core langchain-text-splitters sentence-transformers langchain-huggingface

In [3]:
from langchain_core.documents import Document

### List of Documents

Five documents, covering two course topics (embeddings and chunking).

In [4]:
documents = [
    Document(
        page_content="Embeddings are dense vector representations of text.",
        metadata={
            "source": "file_01.md",
            "page": 1,
            "type": "theory",
            "topic": "embeddings",
            "author": "Author name",
        },
    ),
    Document(
        page_content="Cosine similarity measures the angle between two vectors, not their magnitude.",
        metadata={
            "source": "file_01.md",
            "page": 2,
            "type": "theory",
            "topic": "embeddings",
            "author": "Author name",
        },
    ),
    Document(
        page_content="Chunking is the process of splitting a long document into smaller pieces before generating embeddings.",
        metadata={
            "source": "file_02.md",
            "page": 1,
            "type": "theory",
            "topic": "chunking",
            "author": "Author name",
        },
    ),
    Document(
        page_content="Chunks with overlap repeat part of the text between consecutive pieces, preserving context at the edges.",
        metadata={
            "source": "file_02.md",
            "page": 3,
            "type": "theory",
            "topic": "chunking",
            "author": "Author name",
        },
    ),
    Document(
        page_content="RecursiveCharacterTextSplitter tries to keep whole paragraphs intact before falling back to sentences or characters.",
        metadata={
            "source": "file_02.md",
            "page": 4,
            "type": "example",
            "topic": "chunking",
            "author": "Author name",
        },
    ),
]

### Deliverable 1 — List each document (`page_content` and `metadata`)

In [5]:
for i, document in enumerate(documents, start=1):
    print(f"Document {i}")
    print(f"  page_content: {document.page_content}")
    print(f"  metadata: {document.metadata}")
    print()

Document 1
  page_content: Embeddings are dense vector representations of text.
  metadata: {'source': 'file_01.md', 'page': 1, 'type': 'theory', 'topic': 'embeddings', 'author': 'Author name'}

Document 2
  page_content: Cosine similarity measures the angle between two vectors, not their magnitude.
  metadata: {'source': 'file_01.md', 'page': 2, 'type': 'theory', 'topic': 'embeddings', 'author': 'Author name'}

Document 3
  page_content: Chunking is the process of splitting a long document into smaller pieces before generating embeddings.
  metadata: {'source': 'file_02.md', 'page': 1, 'type': 'theory', 'topic': 'chunking', 'author': 'Author name'}

Document 4
  page_content: Chunks with overlap repeat part of the text between consecutive pieces, preserving context at the edges.
  metadata: {'source': 'file_02.md', 'page': 3, 'type': 'theory', 'topic': 'chunking', 'author': 'Author name'}

Document 5
  page_content: RecursiveCharacterTextSplitter tries to keep whole paragraphs intact 

### Deliverable 2 — Total number of documents

In [6]:
print(f"Total documents: {len(documents)}")

Total documents: 5


### Investigation — What data types does `metadata` accept?

Testing a list and a nested dictionary inside `metadata`.

In [7]:
test_document = Document(
    page_content="Test document to investigate accepted types in metadata.",
    metadata={
        "source": "test.md",
        "keywords": ["rag", "embeddings", "chunking"],  # list
        "review": {"author": "Author name", "approved": True},  # nested dict
    },
)

print(test_document.metadata)
print(type(test_document.metadata["keywords"]))
print(type(test_document.metadata["review"]))

{'source': 'test.md', 'keywords': ['rag', 'embeddings', 'chunking'], 'review': {'author': 'Author name', 'approved': True}}
<class 'list'>
<class 'dict'>


**Result:** `Document` accepts the list and the nested dictionary without
complaint — `metadata` is typed as `dict[str, Any]`, so any serializable
Python value is accepted, including lists and nested dictionaries. Worth
noting, though, that **this is a property of `Document` itself**: when
indexing into a real vector store, many backends require "flat" metadata (no
nested lists/dicts) to support efficient filtering — a concern to address
later in the metadata schema (Exercise 2), not in this exercise.

### Investigation — Document without `metadata`

In [8]:
document_without_metadata = Document(page_content="Document with no explicit metadata.")

print(document_without_metadata.metadata)
print(type(document_without_metadata.metadata))

{}
<class 'dict'>


**Result:** no error — `metadata` defaults to an empty dictionary (`{}`).
The field is optional, so a `Document` without metadata simply carries no
extra information alongside the text (which, in practice, makes it harder to
trace the chunk's origin later — exactly the problem the metadata schema in
Exercise 2 solves).

## Exercise 2 — Designing the Metadata Schema

Schema for the chunks generated in Class 04, with the 7 required minimum
fields plus 3 custom fields, based on what the Class 04 pipeline already
computes (tokens, measured overlap) and what's missing to index without
duplicating content.

### Final Schema

| Field | Description | Origin |
| --- | --- | --- |
| `source` | name of the source `.md` file | required |
| `document_id` | document identifier | required |
| `chunk_index` | chunk's position within the document | required |
| `strategy` | which of the 10 Class 04 strategies produced this chunk | required |
| `chunk_size` | configuration used | required |
| `chunk_overlap` | configuration used | required |
| `char_count` | actual chunk size | required |
| `token_count` | actual chunk size in tokens | **custom** |
| `overlap_percentage` | actual measured overlap with the previous chunk (the result, not the configuration) | **custom** |
| `content_hash` | hash of the chunk's text | **custom** |

### Justification for the Custom Fields

- **`token_count`** — `char_count` doesn't say how much space a chunk
  actually takes up in the LLM's context (the character/token ratio varies
  with language and text density). Class 04 already computes tokens per
  *test*; this is the same data, granular per *chunk*. Answers: "how many
  chunks fit in the context window available for this query?"

- **`overlap_percentage`** — `chunk_overlap` is the *configuration* requested
  from the splitter, not the actual result (LangChain's splitters adjust the
  cut point to respect separators, so the effective overlap varies). This
  field is already computed by `measure_overlap()` in Class 04. Answers:
  "is this chunk likely to bring redundant content if I also retrieve its
  neighbors?"

- **`content_hash`** — since the same document is fragmented by 10 different
  strategies, it's common for two strategies to produce identical or
  near-identical chunk text (e.g. a small "fixed" chunk might coincide with
  a "paragraph" chunk). Answers: "is this chunk already indexed under
  another strategy? is it worth deduplicating before indexing into a real
  vector store?"

### Filled-in Example

In [9]:
import hashlib
import json

example_chunk = {
    "source": "gpt4_technical_report.md",
    "document_id": "gpt4_technical_report",
    "chunk_index": 42,
    "strategy": "fixed_with_overlap",
    "chunk_size": 500,
    "chunk_overlap": 50,
    "char_count": 497,
    "token_count": 118,
    "overlap_percentage": 9.8,
}

chunk_text = (
    "GPT-4 is a large multimodal model that accepts image and text inputs "
    "and produces text outputs. On a suite of professional and academic "
    "benchmarks, GPT-4 exhibits human-level performance..."
)

example_chunk["content_hash"] = hashlib.sha256(chunk_text.encode("utf-8")).hexdigest()[:16]

print(json.dumps(example_chunk, indent=2, ensure_ascii=False))

{
  "source": "gpt4_technical_report.md",
  "document_id": "gpt4_technical_report",
  "chunk_index": 42,
  "strategy": "fixed_with_overlap",
  "chunk_size": 500,
  "chunk_overlap": 50,
  "char_count": 497,
  "token_count": 118,
  "overlap_percentage": 9.8,
  "content_hash": "58260af289f10a4e"
}


### Answers

**Which field would you include to cite the source in the final RAG answer?**

`source` combined with `chunk_index` — together they let you point to
exactly "chunk N of document X" in the answer. A `page` field would be even
more precise, but it's not in the schema because Class 04's extraction
(Docling → Markdown) doesn't preserve page numbers per chunk — that
information would need to be captured during extraction, before chunking,
to be included here truthfully (not estimated).

**Why is `chunk_index` useful, especially if the retrieved chunk is cut off in the middle of an explanation?**

Because it lets you fetch the chunk's direct neighbors (`chunk_index - 1` and
`chunk_index + 1`) within the same `document_id` and `strategy`, to
reconstruct the context around the cut. Without a position index, there's no
way to know that two chunks are neighbors in the original document —
similarity search doesn't guarantee that, since similarity isn't the same
thing as positional proximity in the text.